**Repository release note (2026-10-11):** Leakage-free dataset assembly. Contains historical corrective and repeated cells for 10% VLE features; execute in notebook order. Verify expected exported columns before modelling.

Saved execution outputs were removed from this public copy; source calculations have not been independently rerun. Configure `OULAD_PROJECT_DIR` or use the original Colab directory.


In [ ]:
import pandas as pd
import numpy as np
import os

BASE_PATH = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

DATA_PATH = os.path.join(BASE_PATH, "data")
PROCESSED_PATH = os.path.join(BASE_PATH, "processed")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

print("Notebook 03 environment ready.")
print("Processed folder exists:", os.path.exists(PROCESSED_PATH))

In [ ]:
expected_files = [
    "vle_features_10.csv",
    "vle_features_20.csv",
    "vle_features_30.csv",
    "vle_features_40.csv",
    "assessment_features_10.csv",
    "assessment_features_20.csv",
    "assessment_features_30.csv",
    "assessment_features_40.csv"
]

for file in expected_files:
    path = os.path.join(PROCESSED_PATH, file)
    print(f"{file}: {os.path.exists(path)}")

In [ ]:
for p in [10, 20, 30, 40]:

    vle_temp = pd.read_csv(
        os.path.join(
            PROCESSED_PATH,
            f"vle_features_{p}.csv"
        ),
        nrows=5
    )

    print(f"\n--- VLE {p}% ---")

    feature_names = [
        col for col in vle_temp.columns
        if col not in [
            "code_module",
            "code_presentation",
            "id_student",
            "cutoff_day",
            "final_result"
        ]
    ]

    print(feature_names)

In [ ]:
for p in [10, 20, 30, 40]:

    vle_temp = pd.read_csv(
        os.path.join(
            PROCESSED_PATH,
            f"vle_features_{p}.csv"
        ),
        nrows=1
    )

    required = [
        f"content_clicks_{p}",
        f"forum_clicks_{p}",
        f"navigation_clicks_{p}"
    ]

    missing = [
        col for col in required
        if col not in vle_temp.columns
    ]

    print(f"{p}% cutoff")
    print("Required grouped features:", required)
    print("Missing:", missing)
    print("-" * 50)

In [ ]:
keys = ["code_module", "code_presentation", "id_student"]

# Load existing validated 10% VLE dataset
vle_10_fix = pd.read_csv(
    os.path.join(PROCESSED_PATH, "vle_features_10.csv")
)

# Registration timing
registration = pd.read_csv(
    os.path.join(DATA_PATH, "studentRegistration.csv")
)

# VLE activity-type dictionary
vle_info = pd.read_csv(
    os.path.join(DATA_PATH, "vle.csv")
)

# Attach registration date to eligible 10% students
lookup_10 = vle_10_fix[
    keys + ["cutoff_day"]
].merge(
    registration[
        keys + ["date_registration"]
    ],
    on=keys,
    how="left"
)

# Universal activity grouping chosen earlier
activity_group_map = {
    "oucontent": "content",
    "subpage": "content",
    "resource": "content",
    "forumng": "forum",
    "homepage": "navigation"
}

vle_map = vle_info[
    [
        "code_module",
        "code_presentation",
        "id_site",
        "activity_type"
    ]
].drop_duplicates()

vle_path = os.path.join(DATA_PATH, "studentVle.csv")

group_parts = []

for chunk in pd.read_csv(
    vle_path,
    chunksize=500_000
):

    merged = chunk.merge(
        lookup_10,
        on=keys,
        how="inner"
    )

    # STRICT leakage-free 10% window
    inc = merged[
        (merged["date"] >= 0) &
        (merged["date"] >= merged["date_registration"]) &
        (merged["date"] <= merged["cutoff_day"])
    ].copy()

    if len(inc) == 0:
        continue

    inc = inc.merge(
        vle_map,
        on=[
            "code_module",
            "code_presentation",
            "id_site"
        ],
        how="left"
    )

    inc["activity_group"] = (
        inc["activity_type"]
        .map(activity_group_map)
    )

    inc = inc[
        inc["activity_group"].notna()
    ]

    if len(inc) > 0:
        group_parts.append(
            inc.groupby(
                keys + ["activity_group"],
                as_index=False
            )["sum_click"].sum()
        )

print("10% activity groups processed.")

In [ ]:
group_clicks_10 = pd.concat(
    group_parts,
    ignore_index=True
)

group_clicks_10 = (
    group_clicks_10
    .groupby(
        keys + ["activity_group"],
        as_index=False
    )["sum_click"]
    .sum()
)

group_clicks_10 = (
    group_clicks_10
    .pivot_table(
        index=keys,
        columns="activity_group",
        values="sum_click",
        fill_value=0
    )
    .reset_index()
)

group_clicks_10.columns.name = None

group_clicks_10 = group_clicks_10.rename(
    columns={
        "content": "content_clicks_10",
        "forum": "forum_clicks_10",
        "navigation": "navigation_clicks_10"
    }
)

group_clicks_10.head()

In [ ]:
vle_10_fixed = vle_10_fix.merge(
    group_clicks_10,
    on=keys,
    how="left"
)

new_group_features = [
    "content_clicks_10",
    "forum_clicks_10",
    "navigation_clicks_10"
]

vle_10_fixed[new_group_features] = (
    vle_10_fixed[new_group_features]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(vle_10_fixed))

print(
    "Duplicates:",
    vle_10_fixed.duplicated(
        subset=keys
    ).sum()
)

print("\nMissing grouped features:")
print(
    vle_10_fixed[
        new_group_features
    ].isna().sum()
)

In [ ]:
vle_10_fixed.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "vle_features_10.csv"
    ),
    index=False
)

print("Corrected 10% VLE file saved.")

In [ ]:
for p in [10, 20, 30, 40]:

    vle_temp = pd.read_csv(
        os.path.join(
            PROCESSED_PATH,
            f"vle_features_{p}.csv"
        ),
        nrows=1
    )

    required = [
        f"content_clicks_{p}",
        f"forum_clicks_{p}",
        f"navigation_clicks_{p}"
    ]

    missing = [
        col for col in required
        if col not in vle_temp.columns
    ]

    print(f"{p}% cutoff")
    print("Missing:", missing)
    print("-" * 40)

In [ ]:
student_info = pd.read_csv(
    os.path.join(DATA_PATH, "studentInfo.csv")
)

student_registration = pd.read_csv(
    os.path.join(DATA_PATH, "studentRegistration.csv")
)

print("studentInfo rows:", len(student_info))
print("studentRegistration rows:", len(student_registration))

print("\nstudentInfo columns:")
print(student_info.columns.tolist())

In [ ]:
static_candidates = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability"
]

for col in static_candidates:
    print(f"\n--- {col} ---")
    print("Unique values:", student_info[col].nunique(dropna=False))
    print(student_info[col].value_counts(dropna=False).head(20))

In [ ]:
print(
    student_info[
        static_candidates
    ].isna().sum()
)

In [ ]:
print(
    student_registration["date_registration"]
    .describe()
)

print(
    "\nRegistrations before Day 0:",
    (student_registration["date_registration"] < 0).sum()
)

print(
    "Registrations on/after Day 0:",
    (student_registration["date_registration"] >= 0).sum()
)

In [ ]:
static_audit = pd.DataFrame({
    "Feature": static_candidates,
    "Unique_Values": [
        student_info[col].nunique(dropna=False)
        for col in static_candidates
    ],
    "Missing_Count": [
        student_info[col].isna().sum()
        for col in static_candidates
    ],
    "Missing_Percent": [
        round(student_info[col].isna().mean() * 100, 2)
        for col in static_candidates
    ]
})

static_audit

In [ ]:
for col in [
    "gender",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]:
    print(f"\n--- {col} ---")
    print(student_info[col].value_counts(dropna=False))

In [ ]:
for col in [
    "age_band",
    "disability",
    "region",
    "num_of_prev_attempts",
    "studied_credits"
]:
    print(f"\n--- {col} ---")

    if student_info[col].dtype == "object":
        print(student_info[col].value_counts(dropna=False))
    else:
        print(student_info[col].describe())
        print("\nValue counts:")
        print(student_info[col].value_counts(dropna=False).sort_index().head(30))

In [ ]:
print("--- num_of_prev_attempts ---")
print(student_info["num_of_prev_attempts"].describe())
print(student_info["num_of_prev_attempts"].value_counts().sort_index())

print("\n--- studied_credits ---")
print(student_info["studied_credits"].describe())
print(student_info["studied_credits"].value_counts().sort_index().head(30))

In [ ]:
keys = [
    "code_module",
    "code_presentation",
    "id_student"
]

# Static/background information
static_features = student_info[
    keys + [
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability"
    ]
].copy()

# Add registration timing
static_features = static_features.merge(
    student_registration[
        keys + ["date_registration"]
    ],
    on=keys,
    how="left"
)

# Explicit category for missing deprivation information
static_features["imd_band"] = (
    static_features["imd_band"]
    .fillna("Unknown")
)

print("Static feature rows:", len(static_features))
print(
    "Static duplicates:",
    static_features.duplicated(subset=keys).sum()
)

In [ ]:
vle_10 = pd.read_csv(
    os.path.join(
        PROCESSED_PATH,
        "vle_features_10.csv"
    )
)

assessment_10 = pd.read_csv(
    os.path.join(
        PROCESSED_PATH,
        "assessment_features_10.csv"
    )
)

print("VLE rows:", len(vle_10))
print("Assessment rows:", len(assessment_10))

In [ ]:
assessment_10_merge = assessment_10.drop(
    columns=[
        "cutoff_day",
        "final_result"
    ]
)

dataset_10 = (
    vle_10
    .merge(
        assessment_10_merge,
        on=keys,
        how="left"
    )
    .merge(
        static_features,
        on=keys,
        how="left"
    )
)

print("Complete 10% dataset rows:", len(dataset_10))
print("Columns:", len(dataset_10.columns))

print(
    "Duplicate students:",
    dataset_10.duplicated(subset=keys).sum()
)

In [ ]:
print("Target distribution:")
print(
    dataset_10["final_result"]
    .value_counts()
)

print("\nMissing values:")
missing_10 = (
    dataset_10
    .isna()
    .sum()
)

print(
    missing_10[
        missing_10 > 0
    ]
)

In [ ]:
forbidden_predictors = [
    "date_unregistration"
]

print("Forbidden variables present:")

for col in forbidden_predictors:
    print(
        col,
        col in dataset_10.columns
    )

In [ ]:
print("\nIdentifier/target columns retained only for bookkeeping:")
print(
    dataset_10[
        [
            "code_module",
            "code_presentation",
            "id_student",
            "cutoff_day",
            "final_result"
        ]
    ].head()
)

In [ ]:
keys = [
    "code_module",
    "code_presentation",
    "id_student"
]

# Static/background information
static_features = student_info[
    keys + [
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability"
    ]
].copy()

# Add registration timing
static_features = static_features.merge(
    student_registration[
        keys + ["date_registration"]
    ],
    on=keys,
    how="left"
)

# Explicit category for missing deprivation information
static_features["imd_band"] = (
    static_features["imd_band"]
    .fillna("Unknown")
)

print("Static feature rows:", len(static_features))
print(
    "Static duplicates:",
    static_features.duplicated(subset=keys).sum()
)

In [ ]:
vle_10 = pd.read_csv(
    os.path.join(
        PROCESSED_PATH,
        "vle_features_10.csv"
    )
)

assessment_10 = pd.read_csv(
    os.path.join(
        PROCESSED_PATH,
        "assessment_features_10.csv"
    )
)

print("VLE rows:", len(vle_10))
print("Assessment rows:", len(assessment_10))

In [ ]:
assessment_10_merge = assessment_10.drop(
    columns=[
        "cutoff_day",
        "final_result"
    ]
)

dataset_10 = (
    vle_10
    .merge(
        assessment_10_merge,
        on=keys,
        how="left"
    )
    .merge(
        static_features,
        on=keys,
        how="left"
    )
)

print("Complete 10% dataset rows:", len(dataset_10))
print("Columns:", len(dataset_10.columns))

print(
    "Duplicate students:",
    dataset_10.duplicated(subset=keys).sum()
)

In [ ]:
print("Target distribution:")
print(
    dataset_10["final_result"]
    .value_counts()
)

print("\nMissing values:")
missing_10 = (
    dataset_10
    .isna()
    .sum()
)

print(
    missing_10[
        missing_10 > 0
    ]
)

In [ ]:
forbidden_predictors = [
    "date_unregistration"
]

print("Forbidden variables present:")

for col in forbidden_predictors:
    print(
        col,
        col in dataset_10.columns
    )

In [ ]:
print("\nIdentifier/target columns retained only for bookkeeping:")
print(
    dataset_10[
        [
            "code_module",
            "code_presentation",
            "id_student",
            "cutoff_day",
            "final_result"
        ]
    ].head()
)

In [ ]:
print("=== FINAL 10% DATASET CHECK ===")

print("\nRows:", len(dataset_10))
print("Columns:", len(dataset_10.columns))
print(
    "Duplicates:",
    dataset_10.duplicated(subset=keys).sum()
)

print("\nTarget distribution:")
print(dataset_10["final_result"].value_counts())

print("\nMissing values > 0:")
missing = dataset_10.isna().sum()
print(missing[missing > 0])

print("\nForbidden variable check:")
print(
    "date_unregistration present:",
    "date_unregistration" in dataset_10.columns
)

print("\nStatic variables missing:")
static_check = [
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "date_registration"
]

print(dataset_10[static_check].isna().sum())

In [ ]:
static_check = [
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "date_registration"
]

dataset_10[static_check].isna().sum()

In [ ]:
MODEL_DATA_PATH = os.path.join(
    PROCESSED_PATH,
    "model_datasets"
)

os.makedirs(
    MODEL_DATA_PATH,
    exist_ok=True
)

dataset_10.to_csv(
    os.path.join(
        MODEL_DATA_PATH,
        "leakage_free_10.csv"
    ),
    index=False
)

print("Complete 10% leakage-free dataset saved.")

In [ ]:
def assemble_leakage_free_dataset(progress):

    p = int(progress)

    # ----------------------------
    # Load validated temporal files
    # ----------------------------
    vle = pd.read_csv(
        os.path.join(
            PROCESSED_PATH,
            f"vle_features_{p}.csv"
        )
    )

    assessment = pd.read_csv(
        os.path.join(
            PROCESSED_PATH,
            f"assessment_features_{p}.csv"
        )
    )

    # Assessment contains duplicate bookkeeping columns
    assessment_merge = assessment.drop(
        columns=[
            "cutoff_day",
            "final_result"
        ]
    )

    # ----------------------------
    # Merge all feature families
    # ----------------------------
    dataset = (
        vle
        .merge(
            assessment_merge,
            on=keys,
            how="left"
        )
        .merge(
            static_features,
            on=keys,
            how="left"
        )
    )

    # ----------------------------
    # Integrity checks
    # ----------------------------
    duplicates = (
        dataset
        .duplicated(subset=keys)
        .sum()
    )

    static_missing = (
        dataset[static_check]
        .isna()
        .sum()
        .sum()
    )

    print(f"\n=== {p}% DATASET ===")
    print("Rows:", f"{len(dataset):,}")
    print("Columns:", len(dataset.columns))
    print("Duplicates:", duplicates)
    print(
        "Static missing values:",
        static_missing
    )
    print(
        "date_unregistration present:",
        "date_unregistration" in dataset.columns
    )

    print("\nTarget distribution:")
    print(
        dataset["final_result"]
        .value_counts()
    )

    print("\nMissing variables:")
    missing = dataset.isna().sum()
    print(missing[missing > 0])

    return dataset

In [ ]:
dataset_20 = assemble_leakage_free_dataset(20)

In [ ]:
dataset_30 = assemble_leakage_free_dataset(30)

In [ ]:
dataset_40 = assemble_leakage_free_dataset(40)

In [ ]:
for p, df in {
    10: dataset_10,
    20: dataset_20,
    30: dataset_30,
    40: dataset_40
}.items():

    df.to_csv(
        os.path.join(
            MODEL_DATA_PATH,
            f"leakage_free_{p}.csv"
        ),
        index=False
    )

print("All four leakage-free modelling datasets saved.")

In [ ]:
dataset_summary = []

for p, df in {
    10: dataset_10,
    20: dataset_20,
    30: dataset_30,
    40: dataset_40
}.items():

    dataset_summary.append({
        "Cutoff": f"{p}%",
        "Rows": len(df),
        "Columns": len(df.columns),
        "Pass": (df["final_result"] == "Pass").sum(),
        "Fail": (df["final_result"] == "Fail").sum(),
        "Withdrawn": (df["final_result"] == "Withdrawn").sum(),
        "Distinction": (df["final_result"] == "Distinction").sum(),
        "Duplicates": df.duplicated(subset=keys).sum()
    })

dataset_summary = pd.DataFrame(dataset_summary)

dataset_summary

In [ ]:
dataset_summary.to_csv(
    os.path.join(
        RESULTS_PATH,
        "leakage_free_dataset_summary.csv"
    ),
    index=False
)

print("Dataset summary saved.")

In [ ]:
for p, df in {
    10: dataset_10,
    20: dataset_20,
    30: dataset_30,
    40: dataset_40
}.items():

    registrations = len(df)
    unique_students = df["id_student"].nunique()

    student_counts = (
        df["id_student"]
        .value_counts()
    )

    repeated_students = (
        student_counts > 1
    ).sum()

    print(f"\n--- {p}% cutoff ---")
    print("Registrations:", registrations)
    print("Unique students:", unique_students)
    print(
        "Students appearing in multiple registrations:",
        repeated_students
    )
    print(
        "Maximum registrations for one student:",
        student_counts.max()
    )

In [ ]:
forbidden_terms = [
    "date_unregistration",
    "final_exam",
    "exam_score",
    "withdrawal_date"
]

for p, df in {
    10: dataset_10,
    20: dataset_20,
    30: dataset_30,
    40: dataset_40
}.items():

    found = [
        col for col in df.columns
        if any(
            term.lower() in col.lower()
            for term in forbidden_terms
        )
    ]

    print(f"{p}% forbidden columns:", found)

In [ ]:
for p, df in {
    10: dataset_10,
    20: dataset_20,
    30: dataset_30,
    40: dataset_40
}.items():

    registrations = len(df)
    unique_students = df["id_student"].nunique()

    student_counts = df["id_student"].value_counts()

    repeated_students = (
        student_counts > 1
    ).sum()

    print(f"\n--- {p}% cutoff ---")
    print("Registrations:", registrations)
    print("Unique students:", unique_students)
    print(
        "Students appearing in multiple registrations:",
        repeated_students
    )
    print(
        "Maximum registrations for one student:",
        student_counts.max()
    )